In [12]:
!pip install pandas openpyxl xlrd


   ---------------------------------------- 0/3 [xlrd]
   ---------------------------------------- 0/3 [xlrd]
   ---------------------------------------- 0/3 [xlrd]
   ---------------------------------------- 0/3 [xlrd]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   ---------------------------------------- 3/3 [openpyxl]



In [16]:
import pandas as pd

# CSV file load karein. 
# Agar file encode error day toh encoding='latin1' ya encoding='utf-8' try karein
df = pd.read_csv("resume_data.csv", encoding='utf-8')

# Check karein columns ke names kiya hain
print("--- Dataset Columns ---")
print(df.columns)

# Pehli 5 rows ka text dekhain
print("\n--- First 5 Rows ---")
print(df.head())

--- Dataset Columns ---
Index(['address', 'career_objective', 'skills', 'educational_institution_name',
       'degree_names', 'passing_years', 'educational_results', 'result_types',
       'major_field_of_studies', 'professional_company_names', 'company_urls',
       'start_dates', 'end_dates', 'related_skils_in_job', 'positions',
       'locations', 'responsibilities', 'extra_curricular_activity_types',
       'extra_curricular_organization_names',
       'extra_curricular_organization_links', 'role_positions', 'languages',
       'proficiency_levels', 'certification_providers', 'certification_skills',
       'online_links', 'issue_dates', 'expiry_dates', '﻿job_position_name',
       'educationaL_requirements', 'experiencere_requirement',
       'age_requirement', 'responsibilities.1', 'skills_required',
       'matched_score'],
      dtype='str')

--- First 5 Rows ---
  address                                   career_objective  \
0     NaN  Big data analytics working and database w

In [19]:
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.blank("en")
matcher = PhraseMatcher(nlp.vocab)

# Aapkay project aur academic background kay mutabiq target keywords
skills_list = [
    "FastAPI", "TypeScript", "JavaScript", "Python", "React", "MongoDB", 
    "Machine Learning", "Neural Networks", "Logistic Regression", "KNN", "Naive Bayes",
    "Numerical Methods", "Taylor Polynomials", "Compiler Construction", 
    "Context-Free Grammars", "Directed Acyclic Graphs", "Data Encryption Standard"
]

verbs_list = [
    "Engineered", "Developed", "Deployed", "Implemented", "Designed", 
    "Optimized", "Solved", "Secured", "Managed", "Led"
]

# Patterns ko Matcher mai add karna
skill_patterns = [nlp.make_doc(text) for text in skills_list]
verb_patterns = [nlp.make_doc(text) for text in verbs_list]

matcher.add("SKILL", skill_patterns)
matcher.add("ACTION_VERB", verb_patterns)

TRAIN_DATA = []

# In columns say hum text data nikalain gay training kay liye
columns_to_extract = ['responsibilities', 'skills', 'career_objective', 'skills_required']

for col in columns_to_extract:
    if col in df.columns:
        print(f"Processing column: {col}...")
        for text in df[col].dropna():
            doc = nlp(str(text))
            matches = matcher(doc)
            entities = []
            
            for match_id, start, end in matches:
                label = nlp.vocab.strings[match_id]
                span = doc[start:end]
                entities.append((span.start_char, span.end_char, label))
            
            # Sirf wohi rows save karein jahan keywords milein
            if entities:
                TRAIN_DATA.append((str(text), {"entities": entities}))

print(f"\nTotal training samples successfully created from CSV: {len(TRAIN_DATA)}")

Processing column: responsibilities...
Processing column: skills...
Processing column: career_objective...
Processing column: skills_required...

Total training samples successfully created from CSV: 7695


In [20]:
#from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random

# 1. NER pipeline ko check aur add karna
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)

# 2. Saare unique labels model mai register karna
for _, annotations in TRAIN_DATA:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

# 3. Training configuration
optimizer = nlp.begin_training()
epochs = 10
batch_size = 16

print("🚀 Training started on 7,695 samples...")

for epoch in range(epochs):
    random.shuffle(TRAIN_DATA)
    losses = {}
    
    # Data ko batches mai divide kar kay train karna (Fast approach)
    batches = minibatch(TRAIN_DATA, size=batch_size)
    for batch in batches:
        examples = []
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)
            examples.append(example)
        
        # Model ko pure batch say update karna
        nlp.update(examples, drop=0.3, sgd=optimizer, losses=losses)
        
    print(f"🔥 Epoch {epoch+1}/{epochs} - Loss: {losses['ner']:.4f}")

print("✅ Training completed successfully!")

🚀 Training started on 7,695 samples...
🔥 Epoch 1/10 - Loss: 7085.9731
🔥 Epoch 2/10 - Loss: 0.5190
🔥 Epoch 3/10 - Loss: 26.7701
🔥 Epoch 4/10 - Loss: 15.4876
🔥 Epoch 5/10 - Loss: 0.9554
🔥 Epoch 6/10 - Loss: 13.2753
🔥 Epoch 7/10 - Loss: 12.6858
🔥 Epoch 8/10 - Loss: 0.0000
🔥 Epoch 9/10 - Loss: 0.0000
🔥 Epoch 10/10 - Loss: 0.0000
✅ Training completed successfully!


In [22]:
# Test text jisme humne skills aur verbs dale hain
test_text = "Experienced ML Engineer who engineered deep learning models and deployed APIs using FastAPI and React."

doc = nlp(test_text)

print("--- ATS Model Predictions ---")
if not doc.ents:
    print("No entities detected. Try adding more specific keywords from your skills list.")
else:
    for ent in doc.ents:
        print(f"Detected: '{ent.text}' -------> Label: {ent.label_}")

--- ATS Model Predictions ---
Detected: 'React' -------> Label: SKILL


In [23]:
import os

model_folder = "./ats_cv_model"
if not os.path.exists(model_folder):
    os.makedirs(model_folder)

# Model ko disk par save karna
nlp.to_disk(model_folder)
print(f"💾 Model saved successfully at '{model_folder}'!")
print("Aap ka trained module ab ready hai FastAPI backend par lagne kay liye.")

💾 Model saved successfully at './ats_cv_model'!
Aap ka trained module ab ready hai FastAPI backend par lagne kay liye.
